# Ecommerce AI System - Assignment Notebook
This notebook demonstrates the full churn + marketing AI workflow.

It includes data creation, model training/evaluation, churn prediction for 3 customers, and GPT-generated marketing messages.

## Section 1: Data Generation
In this step, we generate the synthetic churn dataset and save it to data/customers.csv.

In [1]:
from pathlib import Path
import pandas as pd
from src.data_generation import generate_synthetic_data

data_path = Path('data/customers.csv')
df = generate_synthetic_data()
data_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(data_path, index=False)

print(f'Saved dataset to: {data_path}')
print(f'Rows: {len(df)}')
print(f'Columns: {list(df.columns)}')
df.head()

Saved dataset to: data\customers.csv
Rows: 30
Columns: ['purchases_last_month', 'days_since_last_login', 'avg_spend', 'complaints_count', 'churn']


,purchases_last_month,days_since_last_login,avg_spend,complaints_count,churn
0,1,45,28.5,4,1
1,8,5,92.0,0,0
2,3,30,41.8,2,1
3,11,2,135.4,0,0
4,6,10,76.2,1,0


## Section 2: Model Training + Evaluation
We train a Logistic Regression model and evaluate it with Accuracy, Precision, Recall, F1 Score, and ROC-AUC.
We also visualize the ROC curve and show the confusion matrix.

In [ ]:
from src.train_model import train_and_save_model
from src.predict import MODEL_PATH
import matplotlib.pyplot as plt

metrics = train_and_save_model()

print('Classification Metrics')
print('----------------------')
print(f"Accuracy  : {metrics['accuracy']:.3f} -> overall correctness")
print(f"Precision : {metrics['precision']:.3f} -> quality of predicted churners")
print(f"Recall    : {metrics['recall']:.3f} -> how many real churners were detected")
print(f"F1 Score  : {metrics['f1_score']:.3f} -> balance between precision and recall")
print(f"ROC-AUC   : {metrics['roc_auc']:.3f} -> ranking quality across thresholds")
print(f"Model saved to: {MODEL_PATH}")

cm_df = pd.DataFrame(
    metrics['confusion_matrix'],
    index=['Actual 0', 'Actual 1'],
    columns=['Predicted 0', 'Predicted 1']
)

fpr = metrics['roc_curve']['fpr']
tpr = metrics['roc_curve']['tpr']

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"ROC curve (AUC = {metrics['roc_auc']:.3f})")
plt.plot([0, 1], [0, 1], linestyle='--', label='Random baseline')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

cm_df

Accuracy: 1.000
Model saved to: C:\Users\USER\OneDrive\Desktop\IN class GAN\ecommerce_ai_system\models\churn_model.pkl


,Predicted 0,Predicted 1
Actual 0,3,0
Actual 1,0,3


## Section 3: Predictions on 3 Customers
We run the reusable prediction function on three hardcoded customer profiles.
Each output now returns churn probability and a segment label: low, medium, or high risk.

In [ ]:
from src.predict import predict_churn

sample_customers = [
    {'purchases_last_month': 1, 'days_since_last_login': 44, 'avg_spend': 29.8, 'complaints_count': 4},
    {'purchases_last_month': 2, 'days_since_last_login': 10, 'avg_spend': 45.0, 'complaints_count': 2},
    {'purchases_last_month': 11, 'days_since_last_login': 3, 'avg_spend': 141.2, 'complaints_count': 0}
]

prediction_results = []
for idx, customer in enumerate(sample_customers, start=1):
    churn_probability, churn_segment = predict_churn(customer)
    prediction_results.append({
        'customer_no': idx,
        **customer,
        'churn_probability': round(churn_probability, 4),
        'churn_segment': churn_segment
    })

predictions_df = pd.DataFrame(prediction_results)
predictions_df

,customer_no,purchases_last_month,days_since_last_login,avg_spend,complaints_count,predicted_label,churn_probability,churn_risk
0,1,1,44,29.8,4,1,1.0000,high
1,2,2,10,45.0,2,1,0.5501,medium
2,3,11,3,141.2,0,0,0.0000,low


## Section 4: GPT Generated Outputs
We generate marketing emails using a structured prompt with role, tone, and constraints.
The model is asked to return structured JSON with subject and email_body.
If the LLM fails, a risk-based fallback template is returned automatically.

In [ ]:
from src.generate_content import generate_marketing_message

for row in prediction_results:
    customer_data = {
        'purchases_last_month': row['purchases_last_month'],
        'days_since_last_login': row['days_since_last_login'],
        'avg_spend': row['avg_spend'],
        'complaints_count': row['complaints_count']
    }

    print('=' * 90)
    print(
        f"Customer {row['customer_no']} | Prob: {row['churn_probability']:.4f} | "
        f"Segment: {row['churn_segment']}"
    )
    print('-' * 90)

    message = generate_marketing_message(
        customer_data,
        row['churn_segment'],
        row['churn_probability']
    )

    print(f"Subject: {message['subject']}")
    print(message['email_body'])
    print()

Customer 1 | Label: 1 | Prob: 1.0000 | Risk: high
------------------------------------------------------------------------------------------
Subject: Exclusive 20% Off Your Next Purchase

Dear [Customer's Name],

We've missed you at [Brand Name]! It's been a while since your last purchase, and we want to make sure you know how much we value your loyalty.

As a valued customer, we're offering you an exclusive 20% discount on your next purchase. Use code COMEBACK20 at checkout to redeem your discount. This is a one-time offer, and it's only valid for the next 48 hours.

Our entire collection is now 20% off, so you can treat yourself to a new favorite or stock up on the essentials. Don't miss out on this amazing opportunity to save big!

Click the link below to start shopping and redeem your discount:

[Insert CTA button: Shop Now and Save 20%]

Hurry, offer ends soon!

Best regards,
[Your Name]
[Brand Name] Team

Customer 2 | Label: 1 | Prob: 0.5501 | Risk: medium
-----------------------

## Section 5: Reflection
Discriminative models are great at predicting outcomes like churn, but they do not create customer-facing actions by themselves.
Generative models are great at writing personalized content, but they do not reliably decide which customer needs what action.
If we use only discriminative models, we get scores without persuasive communication.
If we use only generative models, we risk sending the wrong message to the wrong customer and wasting budget.
Combining both gives clear business value: the discriminative model finds who is at risk, and the generative model crafts what to say.
In e-commerce, this improves retention, increases campaign efficiency, and creates a better customer experience at scale.